
Week29_Fine_Tuning_BERT

Cell 1  → Install libraries
Cell 2  → Imports
Cell 3  → Load IMDb
Cell 4  → Inspect dataset
Cell 5  → Inspect example
Cell 6  → Select smaller dataset
Cell 7  → Load tokenizer
Cell 8  → Tokenization function
Cell 9  → Tokenize train
Cell 10 → Tokenize test

Cell 12 → Import BERT
Cell 13 → Load BERT
Cell 15 → Labels
Cell 16 → Data collator

Cell 17 → Trainer imports
Cell 18 → TrainingArguments
Cell 19 → Trainer
Cell 20 → trainer.train()

In [ ]:
!pip install -q transformers datasets evaluate accelerate

import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

from transformers import AutoTokenizer, AutoModelForSequenceClassification
model_name = "bert-base-cased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

text = "I love this movie!"
tokens = tokenizer(text)
print(tokens)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)


text = "I love this movie!"
inputs = tokenizer(
    text,
    return_tensor="pt"
)
print(inputs)


In [ ]:

from datasets import load_dataset
from transformers import AutoTokenizer

dataset = load_dataset("stanfordnlp/imdb")

print(dataset)

print(dataset["train"][0])

train_dataset = dataset["train"].shuffle(seed=42).select(range(2000))
test_dataset = dataset["test"].shuffle(seed=42).select(range(500))

model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)


In [ ]:

def tokenize_function(examples):
  return tokenizer(
      examples["text"],
      trunction=True,
      padding="max_length",
      max_length=128
  )

tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True
)

print(tokenized_train[0])


In [ ]:

from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

print(model)


In [ ]:

id2label = {
    0: "NEGATIVE",
    1: "POSITIVE"
}

label2id = {
    "NEGATIVE": 0,
    "POSITIVE": 1
}

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)


In [ ]:

from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

from transformers import TrainingArguments, Trainer

# Correct the tokenizer function to fix the typo: 'truncation=True'
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

# Re-tokenize datasets with correct truncation
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

# Define TrainingArguments
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01
)

tokenized_test = test_dataset.map(
    tokenize_function,
    batched=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    data_collator=data_collator
)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    data_collator=data_collator
)

trainer.train()

results = trainer.evaluate()

print(results)

import evaluate

accuracy = evaluate.load("accuracy")


In [ ]:

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    predictions = predictions.argmax(axis=-1)

    return accuracy.compute(
        predictions=predictions,
        references=labels
    )

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    data_collator=data_collator
)

results = trainer.evaluate()

print(results)

import torch

text = "I really loved this movie. It was amazing!"

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True
)

inputs = {
    key: value.to(model.device)
    for key, value in inputs.items()
}

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits

prediction = torch.argmax(logits, dim=-1).item()

print("Prediction:", prediction)
print("Label:", id2label[prediction])


In [ ]:

text = "This movie was terrible and boring."

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True
)

inputs = {
    key: value.to(model.device)
    for key, value in inputs.items()
}

with torch.no_grad():
    outputs = model(**inputs)

prediction = torch.argmax(
    outputs.logits,
    dim=-1
).item()

print("Prediction:", prediction)
print("Label:", id2label[prediction])





In [ ]:



save_path = "./my_fine_tuned_bert"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

import os

print(os.listdir(save_path))


In [ ]:

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

saved_model = AutoModelForSequenceClassification.from_pretrained(
    "./my_fine_tuned_bert"
)

saved_tokenizer = AutoTokenizer.from_pretrained(
    "./my_fine_tuned_bert"
)



text = "I enjoyed this movie so much!"

inputs = saved_tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True
)

inputs = {
    key: value.to(saved_model.device)
    for key, value in inputs.items()
}

with torch.no_grad():
    outputs = saved_model(**inputs)

prediction = torch.argmax(
    outputs.logits,
    dim=-1
).item()

print("Prediction:", prediction)
print("Label:", id2label[prediction])

texts = [
    "This movie was fantastic!",
    "I hated this movie.",
    "The movie was boring and terrible.",
    "I really enjoyed this film.",
    "Amazing acting and a great story!"
]

for text in texts:

    inputs = saved_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    inputs = {
        key: value.to(saved_model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = saved_model(**inputs)

    prediction = torch.argmax(
        outputs.logits,
        dim=-1
    ).item()

    print(text)
    print("→", id2label[prediction])
    print()


In [ ]:
!zip -r my_fine_tuned_bert.zip my_fine_tuned_bert



                 WEEK 29
                    │
                    ▼
          Pre-trained BERT
                    │
                    ▼
             IMDb Dataset
                    │
                    ▼
                Tokenizer
                    │
                    ▼
             Tokenized Data
                    │
                    ▼
                BERT Model
                    │
                    ▼
                 Trainer
                    │
                    ▼
              Fine-Tuning
                    │
                    ▼
                Evaluation
                    │
                    ▼
                 Accuracy
                    │
                    ▼
             Test New Text
                    │
                    ▼
               Save Model
                    │
                    ▼
             Load Model Later